# Image Captioning — Kaggle Training Run

**ResNet50 frozen encoder + LSTM decoder on COCO Karpathy split.**

This notebook is a thin launcher: it clones the repo, installs the package, mounts data
from Kaggle Datasets, runs training with auto-resume, then pushes the best checkpoint
to HuggingFace Hub and runs full `pycocoevalcap` evaluation.

---
**PRE-REQUISITES (one-time, done before running this notebook):**

1. A Kaggle Dataset called `coco-features` containing the cached encoder features
   (fp16 .npy files per image, produced by `scripts/cache_features.py`).
2. A Kaggle Dataset `coco-karpathy` with `dataset_coco.json`.
3. The standard `coco-2017` Dataset (or your own COCO images + annotations).
4. A HuggingFace Hub token saved as a Kaggle Secret (`HF_TOKEN`).
5. The vocab.json uploaded to the `coco-features` Dataset.

**How to create the cached features Dataset:**
    Run `notebooks/cache_features_kaggle.ipynb` first (included in repo).

---

## 1. Setup — repo, env, and data mounts

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/MohamedShakshak/Image-Captioning.git"
CONFIG_PATH = "configs/default.yaml"

def _inside_repo() -> bool:
    """Check if current directory is the repo root."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, timeout=5,
        )
        return out.returncode == 0 and out.stdout.strip().endswith("Image-Captioning")
    except Exception:
        return False

if _inside_repo():
    print("Already inside the repo — skipping clone.")
elif os.path.isdir("Image-Captioning"):
    print("Repo directory exists — cd into it.")
    %cd Image-Captioning
else:
    !git clone --depth 1 {REPO_URL}
    %cd Image-Captioning

# Install the package + deps
!pip install -e . -q
!pip install -e .[app] -q

# Ensure src/ is on sys.path (pip's .pth may not be re-read by the kernel)
import sys, os
_src = os.path.join(os.getcwd(), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
print(f"Python: {sys.version}")
import torch; print(f"Torch: {torch.__version__}")
import torchvision; print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Kaggle Datasets — these must be added to the notebook before running.
# From the Kaggle UI: Add Data -> select coco-2017, coco-karpathy, coco-features

COCO_ROOT = "/kaggle/input/coco-2017"
KARPATHY_SPLIT = "/kaggle/input/coco-karpathy/dataset_coco.json"
FEATURES_DIR = "/kaggle/input/coco-features"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
VOCAB_PATH = f"{FEATURES_DIR}/vocab.json"

# Verify mounts exist
for p in [COCO_ROOT, KARPATHY_SPLIT, FEATURES_DIR]:
    assert os.path.exists(p), f"Missing mount: {p}"
    print(f"OK: {p}")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Training

Runs the main training loop. Auto-resumes from the latest checkpoint if one exists.

In [ ]:
import sys
sys.argv = [
    "train.py",
    "--config", CONFIG_PATH,
    f"--data.coco-root={COCO_ROOT}",
    f"--data.karpathy-split={KARPATHY_SPLIT}",
    f"--data.features-dir={FEATURES_DIR}",
    f"--data.vocab-path={VOCAB_PATH}",
    f"--checkpoint.dir={CHECKPOINT_DIR}",
    "--checkpoint.save-latest=true",
    "--checkpoint.save-best=true",
    "--train.amp=false",      # P100 no tensor cores; LSTM + fp16 = NaN risk
]

from train import train as run_training
from config import parse_args

cfg, _ = parse_args()
print("Config:", cfg)
run_training(cfg)

## 3. Full Evaluation on Karpathy Test Split

Runs `pycocoevalcap` with all 5 metrics (BLEU-1..4, CIDEr, ROUGE-L, METEOR).
Java is pre-installed on Kaggle, so CIDEr and METEOR work.

In [ ]:
sys.argv = [
    "evaluate.py",
    "--config", CONFIG_PATH,
    f"--data.coco-root={COCO_ROOT}",
    f"--data.karpathy-split={KARPATHY_SPLIT}",
    f"--data.features-dir={FEATURES_DIR}",
    f"--data.vocab-path={VOCAB_PATH}",
    f"--checkpoint.dir={CHECKPOINT_DIR}",
    "--eval.full-metrics=true",  # enables CIDEr + METEOR (Java required)
    "--eval.beam-size=3",
]

from evaluate import evaluate as run_eval

cfg, _ = parse_args()
run_eval(cfg)

## 4. Save results and plot curves

In [ ]:
# Generate training curves
!python scripts/plot_curves.py --metrics metrics.json --out /kaggle/working/curves.png

# Copy results to working dir for download
import shutil
for f in ["metrics.json", "eval_out/preds.json", "eval_out/gts.json", "curves.png"]:
    if os.path.exists(f):
        shutil.copy(f, "/kaggle/working/")
        print(f"Saved /kaggle/working/{f}")

## 5. Push to HuggingFace Hub

Pushes the best checkpoint + vocab to HF Hub (read by the Streamlit app via `Captioner.from_pretrained`).
Requires a Kaggle Secret `HF_TOKEN` with a write-capable HF token.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "MohamedShakshak/image-captioning-pytorch"

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    # Ensure repo exists
    try:
        api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
    except Exception as e:
        print(f"Repo may already exist: {e}")

    # Upload best checkpoint
    best_path = f"{CHECKPOINT_DIR}/best.pt"
    if os.path.exists(best_path):
        api.upload_file(
            path_or_fileobj=best_path,
            path_in_repo="best.pt",
            repo_id=HF_REPO,
            repo_type="model",
        )
        print(f"Uploaded best.pt to {HF_REPO}")

    # Upload vocab (needed by Captioner.from_pretrained)
    if os.path.exists(VOCAB_PATH):
        api.upload_file(
            path_or_fileobj=VOCAB_PATH,
            path_in_repo="vocab.json",
            repo_id=HF_REPO,
            repo_type="model",
        )
        print(f"Uploaded vocab.json to {HF_REPO}")

    # Upload eval results
    for f in ["preds.json", "gts.json", "curves.png"]:
        p = f"/kaggle/working/{f}"
        if os.path.exists(p):
            api.upload_file(
                path_or_fileobj=p,
                path_in_repo=f"eval/{f}",
                repo_id=HF_REPO,
                repo_type="model",
            )
            print(f"Uploaded eval/{f}")
else:
    print("No HF_TOKEN found — skip upload. Results remain in /kaggle/working/.")

## 6. Summary

Check `/kaggle/working/` for:
- `checkpoints/best.pt` — best model weights
- `checkpoints/latest.pt` — latest model weights (for resume)
- `metrics.json` — per-epoch train/val loss + LR
- `curves.png` — training curve plot
- `eval_out/preds.json`, `eval_out/gts.json` — predictions + ground truth

The best checkpoint has been pushed to HuggingFace Hub at `MohamedShakshak/image-captioning-pytorch`.

To download results: go to the Kaggle notebook -> Output tab -> Download.

In [ ]:
print("✅ Training run complete.")
!ls -lh /kaggle/working/